<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>Teoría de Redes Neuronales Recurrentes</h2>
</div>

## Introducción


Los humanos no comienzan su pensamiento desde cero cada segundo. A medida que leemos algún artículo, comprendemos cada palabra en función de su comprensión de las palabras anteriores. Nuestros pensamientos tienen persistencia.

Las redes neuronales tradicionales no pueden hacer esto y parece una gran deficiencia. Por ejemplo, imagine que desea clasificar qué tipo de evento está sucediendo en cada punto de una película. No está claro cómo una red neuronal tradicional podría usar su razonamiento sobre eventos anteriores en la película para informar los posteriores.

Las redes neuronales recurrentes abordan este problema. Son redes con bucles en ellas, lo que permite que la información persista.


<img src="Figures/RNN-rolled.png" alt="Grayscale Image" width="100">


En el diagrama anterior, una parte de la red neuronal, $A$, analiza alguna entrada $x_t$ y genera un valor $h_t$. Un bucle permite que la información pase de un paso de la red al siguiente.

Estos bucles hacen que las redes neuronales recurrentes parezcan algo misteriosas. Sin embargo, si piensa un poco más, resulta que no son tan diferentes de una red neuronal normal. Se puede pensar en una red neuronal recurrente como múltiples copias de la misma red, cada una de las cuales pasa un mensaje a un sucesor. Considere lo que sucede si desenrollamos el ciclo:

<img src="Figures/RNN-unrolled.png" alt="Grayscale Image" width="500">

Esta naturaleza revela que las redes neuronales recurrentes están íntimamente relacionadas con secuencias y listas. Son la arquitectura natural de la red neuronal para usar para tales datos.

¡Y ciertamente se usan! En los últimos años, ha habido un éxito increíble al aplicar RNN a una variedad de problemas: -  

- reconocimiento de voz
- modelado de lenguaje
- traducción
- subtítulos de imágenes... La lista continúa. 

Las "LSTM" son un tipo muy especial de red neuronal recurrente que funciona, para muchas tareas, mucho mejor que la versión estándar. Casi todos los resultados emocionantes basados en redes neuronales recurrentes se logran con ellos. 

## El problema de las dependencias a largo plazo

Uno de los atractivos de las RNN es la idea de que podrían conectar información previa con la tarea actual, como el uso de cuadros de video anteriores que podrían informar la comprensión del cuadro actual. Si los RNN pudieran hacer esto, serían extremadamente útiles. Pero pueden? Eso depende.

A veces, solo necesitamos mirar información reciente para realizar la tarea actual. Por ejemplo, considere un modelo de lenguaje que intente predecir la siguiente palabra en función de las anteriores. Si estamos tratando de predecir la última palabra en "las nubes están en el cielo", no necesitamos más contexto: es bastante obvio que la siguiente palabra será cielo. En tales casos, donde la brecha entre la información relevante y el lugar donde se necesita es pequeña, los RNN pueden aprender a usar la información pasada.

<img src="Figures/RNN-shorttermdepdencies.png" alt="Grayscale Image" width="300">

Pero también hay casos en los que necesitamos más contexto. Considere tratar de predecir la última palabra en el texto "Crecí en Francia... hablo francés con fluidez". Información reciente sugiere que la siguiente palabra probablemente sea el nombre de un idioma, pero si queremos acotar qué idioma, necesitamos el contexto de Francia, desde más atrás. Es completamente posible que la brecha entre la información relevante y el punto donde se necesita se vuelva muy grande.

Desafortunadamente, a medida que crece esa brecha, los RNN se vuelven incapaces de aprender a conectar la información.

<img src="Figures/RNN-longtermdependencies.png" alt="Grayscale Image" width="400">

En teoría, los RNN son absolutamente capaces de manejar tales "dependencias a largo plazo". Un ser humano podría elegir cuidadosamente los parámetros para resolver problemas de juguetes de esta forma. Lamentablemente, en la práctica, los RNN no parecen poder aprenderlos. El problema fue explorado en profundidad por Hochreiter (1991) y Bengio, et al. (1994), quien encontró algunas razones bastante fundamentales por las que podría ser difícil.

¡Afortunadamente, los LSTM no tienen este problema!

## Redes LSTM 

Las redes de memoria a largo plazo, generalmente llamadas simplemente "LSTM", son un tipo especial de RNN, capaces de aprender dependencias a largo plazo. Fueron introducidos por Hochreiter & Schmidhuber (1997), y muchas personas los refinaron y popularizaron en trabajos posteriores.1 Funcionan tremendamente bien en una gran variedad de problemas y ahora se usan ampliamente.

Los LSTM están diseñados explícitamente para evitar el problema de dependencia a largo plazo. Recordar información durante largos períodos de tiempo es prácticamente su comportamiento predeterminado, ¡no es algo que les cueste aprender!

Todas las redes neuronales recurrentes tienen la forma de una cadena de módulos repetidos de red neuronal. En RNN estándar, este módulo repetitivo tendrá una estructura muy simple, como una sola capa de tanh.

<img src="Figures/LSTM3-SimpleRNN.png" alt="Grayscale Image" width="400">

Los LSTM también tienen esta estructura similar a una cadena, pero el módulo repetitivo tiene una estructura diferente. En lugar de tener una sola capa de red neuronal, hay cuatro que interactúan de una manera muy especial.

<img src="Figures/LSTM3-chain.png" alt="Grayscale Image" width="400">


### La idea central detrás de los LSTM

La clave de los LSTM es el estado de la celda, la línea horizontal que atraviesa la parte superior del diagrama.

El estado de la celda es como una cinta transportadora. Se ejecuta directamente a lo largo de toda la cadena, con solo algunas interacciones lineales menores. Es muy fácil que la información fluya sin cambios.

<img src="Figures/LSTM3-C-line.png" alt="Grayscale Image" width="500">

El LSTM tiene la capacidad de eliminar o agregar información al estado de la celda, cuidadosamente regulado por estructuras llamadas puertas.

Las puertas son una forma de dejar pasar información opcionalmente. Están compuestos por una capa de red neuronal sigmoidea y una operación de multiplicación puntual.

<img src="Figures/LSTM3-gate.png" alt="Grayscale Image" width="100">



La capa sigmoide genera números entre cero y uno, que describen la cantidad de cada componente que se debe dejar pasar. Un valor de cero significa "no dejar pasar nada", mientras que un valor de uno significa "¡dejar pasar todo!"

Un LSTM tiene tres de estas puertas para proteger y controlar el estado de la celda.

## Recorrido paso a paso de LSTM

El primer paso en nuestro LSTM es decidir qué información vamos a desechar del estado de la celda. Esta decisión la toma una capa sigmoide llamada **"capa de puerta de olvido"**. Mira $h_{t−1}$ y $x_t$, y genera un número entre 0 y 1 para cada número en el estado de celda $C_{t−1}$. Un 1 representa "mantener esto por completo", mientras que un 0 representa "deshacerse de esto por completo".

Volvamos a nuestro ejemplo de un modelo de lenguaje que intenta predecir la siguiente palabra basándose en todas las anteriores. En tal problema, el estado de la celda podría incluir el género del sujeto presente, de modo que se puedan usar los pronombres correctos. Cuando vemos un tema nuevo, queremos olvidar el género del tema anterior.

<img src="Figures/LSTM3-focus-f.png" alt="Grayscale Image" width="600">

El siguiente paso es decidir qué nueva información vamos a almacenar en el estado de la celda. Esto tiene dos partes. Primero, una capa sigmoidea llamada "capa de puerta de entrada" decide qué valores actualizaremos. Luego, una capa tanh crea un vector de nuevos valores candidatos, $C~t$, que podrían agregarse al estado. En el siguiente paso, combinaremos estos dos para crear una actualización del estado.

En el ejemplo de nuestro modelo de lenguaje, nos gustaría agregar el género del nuevo sujeto al estado de la celda, para reemplazar el anterior que estamos olvidando.

<img src="Figures/LSTM3-focus-i.png" alt="Grayscale Image" width="600">

Ahora es el momento de actualizar el estado de celda anterior, $C_{t−1}$, al nuevo estado de celda $C_t$. Los pasos anteriores ya decidieron qué hacer, solo necesitamos hacerlo.

Multiplicamos el estado anterior por $f_t$, olvidando las cosas que decidimos olvidar antes. Luego lo sumamos $i_t * \tilde{C}_t$. Estos son los nuevos valores candidatos, escalados por cuánto decidimos actualizar cada valor de estado.

En el caso del modelo de lenguaje, aquí es donde descartaríamos la información sobre el género del sujeto anterior y agregaríamos la nueva información, como decidimos en los pasos anteriores.

<img src="Figures/LSTM3-focus-C.png" alt="Grayscale Image" width="600">

Finalmente, tenemos que decidir qué vamos a generar. Esta salida se basará en nuestro estado de celda, pero será una versión filtrada. Primero, ejecutamos una capa sigmoidea que decide qué partes del estado de la celda vamos a generar. Luego, ponemos el estado de la celda a través de tanh (para que los valores estén entre −1 y 1) y lo multiplicamos por la salida de la puerta sigmoidea, de modo que solo emitamos las partes que decidimos.

Para el ejemplo del modelo de lenguaje, dado que acaba de ver un sujeto, es posible que desee generar información relevante para un verbo, en caso de que eso sea lo siguiente. Por ejemplo, podría mostrar si el sujeto es singular o plural, para que sepamos en qué forma se debe conjugar un verbo si eso es lo que sigue a continuación.

<img src="Figures/LSTM3-focus-o.png" alt="Grayscale Image" width="600">

### Variantes de la memoria a largo plazo

Lo que he descrito hasta ahora es un LSTM bastante normal. Pero no todos los LSTM son iguales a los anteriores. De hecho, parece que casi todos los documentos que involucran LSTM usan una versión ligeramente diferente. Las diferencias son menores, pero vale la pena mencionar algunas de ellas.

Una variante popular de LSTM, presentada por Gers & Schmidhuber (2000), agrega "conexiones de mirilla". Esto significa que dejamos que las capas de la puerta miren el estado de la celda.

<img src="Figures/LSTM3-var-peepholes.png" alt="Grayscale Image" width="600">

El diagrama de arriba agrega mirillas a todas las puertas, pero muchos papeles darán algunas mirillas y otras no.

Otra variación es utilizar puertas de entrada y de olvido acopladas. En lugar de decidir por separado qué olvidar y qué debemos agregar nueva información, tomamos esas decisiones juntos. Solo olvidamos cuando vamos a ingresar algo en su lugar. Solo ingresamos nuevos valores al estado cuando olvidamos algo más antiguo.

<img src="Figures/LSTM3-var-tied.png" alt="Grayscale Image" width="600">

Una variación un poco más dramática del LSTM es la Unidad Recurrente Cerrada, o GRU, presentada por Cho, et al. (2014). Combina las puertas de entrada y de olvido en una sola "puerta de actualización". También fusiona el estado de la celda y el estado oculto, y realiza algunos otros cambios. El modelo resultante es más simple que los modelos LSTM estándar y se ha vuelto cada vez más popular.

<img src="Figures/LSTM3-var-GRU.png" alt="Grayscale Image" width="600">

Estas son solo algunas de las variantes de LSTM más notables. Hay muchos otros, como los RNN controlados por profundidad de Yao, et al. (2015). También existe un enfoque completamente diferente para abordar las dependencias a largo plazo, como Clockwork RNN de Koutnik, et al. (2014).

¿Cuál de estas variantes es mejor? ¿Importan las diferencias? Greff, et al. (2015) hacen una buena comparación de variantes populares y descubren que todas son casi iguales. Jozefowicz, et al. (2015) probaron más de diez mil arquitecturas RNN y encontraron algunas que funcionaron mejor que las LSTM en ciertas tareas.
